Project : Holiday weather

There is nothing I like better than taking a holiday. In this project I am going to use the historic weather data from the Weather Underground for London to try to predict two good weather weeks to take off as holiday. Of course the weather in the summer of 2025 may be very different to 2020 but it should give some indication of when would be a good time to take a summer break.

Getting the data
Weather Underground keeps historical weather data collected in many airports around the world. Right-click on the following URL and choose 'Open Link in New Window' (or similar, depending on your browser):

http://www.wunderground.com/history

When the new page opens start typing 'LHR' in the 'Location' input box and when the pop up menu comes up with the option 'LHR, United Kingdom' select it and then click on 'Submit'.

When the next page opens with London Heathrow data, click on the 'Custom' tab and select the time period From: 1 January 2023 to: 31 December 2023 and then click on 'Get History'. The data for that year should then be displayed further down the page.

You can copy each month's data directly from the browser to a text editor like Notepad or TextEdit, to obtain a single file with as many months as you wish.

Now load the CSV file into a dataframe making sure that any extra spaces are skipped:

In [5]:
# Load the heathrow weather data
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
london_heathrow = pd.read_csv('https://raw.githubusercontent.com/Solomakinde10/Heathrow-Weather-data/refs/heads/main/london_Heathrow.csv', skipinitialspace= True)
london_heathrow.head()

# Cnovert the Date column to datetime
london_heathrow["Date"] = pd.to_datetime(london_heathrow["Date"], dayfirst=True, errors="coerce")

# Drop the precipitation column (all values were nil)
london_heathrow = london_heathrow.drop("Precipitation (in)", axis =1)
london_heathrow.describe()

,Max Temperature (°F),Avg Temperature (°F),Min Temperature (°F),Max Dew Point (°F),Avg Dew Point (°F),Min Dew Point (°F),Max Humidity (%),Avg Humidity (%),Min Humidity (%),Max Wind Speed (mph),Avg Wind Speed (mph),Min Wind Speed (mph),Max Pressure (in),Avg Pressure (in),Min Pressure (in)
count,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000,366.000000
mean,65.079235,57.761202,50.622951,52.797814,48.703825,44.174863,92.024590,73.815027,53.603825,14.928962,8.968579,3.625683,29.981967,29.895628,29.803552
std,11.540839,10.185833,10.028448,8.646963,8.709927,9.341439,6.119087,8.788465,12.764844,4.906845,3.596242,2.724681,0.271379,0.292055,0.318323
min,34.000000,27.800000,0.000000,28.000000,24.900000,0.000000,69.000000,49.300000,0.000000,5.000000,1.400000,0.000000,29.100000,29.000000,28.900000
25%,57.000000,50.325000,45.000000,46.500000,42.900000,37.000000,88.000000,68.225000,44.000000,12.000000,6.300000,2.000000,29.900000,29.700000,29.700000
50%,66.000000,60.000000,54.000000,54.000000,50.300000,46.000000,94.000000,73.500000,52.000000,14.000000,8.500000,3.000000,30.000000,29.900000,29.800000
75%,73.000000,65.175000,59.000000,60.500000,55.700000,50.000000,94.000000,80.475000,63.750000,17.000000,11.000000,5.000000,30.100000,30.100000,30.000000
max,90.000000,80.000000,72.000000,68.000000,63.200000,61.000000,100.000000,95.300000,88.000000,32.000000,23.100000,16.000000,30.800000,30.800000,30.600000


Set Date as the Index

In [ ]:
# Set Date as the index for easier filtering
london_heathrow.index = london_heathrow["Date"]
london_heathrow.head()

Step-2 Finding a summer break

According to meteorologists, summer extends for the whole months of June, July, and August in the northern hemisphere and the whole months of December, January, and February in the southern hemisphere. So create a dataframe that holds just those months using the datetime index.

In [ ]:
# Filtering for summer months
summer_data = london_heathrow.loc[datetime(2023,6,1) : datetime(2023,8,30)].copy()
summer_data


Converting Farenheit to Celsius

In [ ]:
# Convert Fahrenheit to Celsius for each temperature column

summer_data['Max Temperature (°C)'] = (summer_data['Max Temperature (°F)'] - 32) * 5/9
summer_data['Avg Temperature (°C)'] = (summer_data['Avg Temperature (°F)'] - 32) * 5/9
summer_data['Min Temperature (°C)'] = (summer_data['Min Temperature (°F)'] - 32) * 5/9

summer_data['Max Dew Point (°C)'] = (summer_data['Max Dew Point (°F)'] - 32) * 5/9
summer_data['Avg Dew Point (°C)'] = (summer_data['Avg Dew Point (°F)'] - 32) * 5/9
summer_data['Min Dew Point (°C)'] = (summer_data['Min Dew Point (°F)'] - 32) * 5/9

# Remove the old Fahrenheit columns
summer_data = summer_data.drop(['Max Temperature (°F)', 'Avg Temperature (°F)', 'Min Temperature (°F)',
                                 'Max Dew Point (°F)', 'Avg Dew Point (°F)', 'Min Dew Point (°F)'], axis=1)

# Check the result
summer_data

Weekly Aggregation: To identify the best weeks for a holiday, the summer data is grouped into weekly averages

In [ ]:
weekly_data = summer_data.set_index('Date').resample('W').mean().reset_index()
weekly_data

Creating a class "Weather Score" to rank the weeks

The scoring system based on ideal weather conditions for a comfortable summer holiday. Each metric contributes to a total score out of 9 points:

In [ ]:
def weather_score(row):
    score = 0

    # Temperature score
    temp_score = max(0, 3 - abs(22 - row['Avg Temperature (°C)']) / 4)
    score += temp_score
    if row['Max Temperature (°C)'] <= 28:
         score += 1

    # Humidity Score
    humidity_score = max(0, 2 - abs(50 - row['Avg Humidity (%)']) / 15)
    score += humidity_score

    # Dew point score
    dew_point_score = max(0, 2 - abs(13 - row['Avg Dew Point (°C)']) / 6)
    score += dew_point_score

    # Wind speed score
    if 5 <= row['Avg Wind Speed (mph)'] <= 15:
        score += 1

    return round(score, 1)

weekly_data['Score'] = weekly_data.apply(weather_score, axis=1)

In [ ]:
# Displaying the weekly_data dataframe with the weather score
weekly_data

Displaying the weekly score in a barchart


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(weekly_data['Date'], weekly_data['Score'], color='blue')
plt.title('Weekly Comfort Score')
plt.xlabel('Week')
plt.ylabel('Comfort Score')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

The analysis reveals that the best period for a London summer holiday is June 12th to June 25th, 2023, with comfort scores of 7.8 and 7.6 respectively.